# TN-SHAP-G Quickstart

This notebook demonstrates the complete TN-SHAP-G pipeline:

1. **Create a masked game** from graph features and a teacher model
2. **Train a TN surrogate** that approximates the game
3. **Compute deterministic Shapley values** via diagonal derivative integral
4. **Compute O2 interactions** on graph edges
5. **Sanity check** against exact enumeration

**Expected runtime: 3-5 minutes on CPU**

## Setup

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import torch
import networkx as nx

# Add src to path if running from notebooks directory
sys.path.insert(0, str(Path.cwd().parent / "src"))

from tnshapg import (
    # Core components
    GraphAlignedTN,
    MaskedGame,
    MLPTeacher,
    # Training
    train_surrogate,
    TrainingConfig,
    # Shapley computation
    compute_diagonal_shapley,
    compute_o2_interactions,
    # Exact enumeration for comparison
    exact_shapley_fast,
    efficiency_check,
    compute_o2_exact,
    # Utilities
    set_seed,
    cosine_similarity,
    max_abs_error,
)

print("TN-SHAP-G loaded successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Configuration
N_NODES = 12          # Number of nodes (keep small for exact enumeration)
N_FEATURES = 7        # Feature dimension
BOND_DIM = 4          # TN bond dimension
N_TRAIN_SAMPLES = 500 # Training coalitions
M_INTERP = 16         # Interpolation nodes
SEED = 42

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Step 1: Create Synthetic Graph

In [ ]:
# Create a random tree graph
G = nx.random_tree(N_NODES, seed=SEED)

# Add a few extra edges for interest
np.random.seed(SEED)
nodes = list(G.nodes())
for _ in range(2):
    u, v = np.random.choice(nodes, 2, replace=False)
    if not G.has_edge(u, v):
        G.add_edge(u, v)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Create random node features
X = torch.randn(N_NODES, N_FEATURES, dtype=torch.float32)
baseline = torch.zeros_like(X)  # Zero baseline

print(f"Node features: {X.shape}")
print(f"Feature range: [{X.min():.2f}, {X.max():.2f}]")

## Step 2: Create Teacher Model and Game

In [ ]:
# Create MLP teacher model
# This aggregates node features and applies a small MLP
teacher = MLPTeacher(
    n_features=N_FEATURES,
    hidden_dims=[32, 16],
    aggregation="sum",
    seed=SEED,
)
teacher = teacher.to(device)
teacher.eval()

# Create masked game
game = MaskedGame(
    X=X,
    baseline=baseline,
    teacher=teacher,
    device=device,
)

# Check game values at extremes
v_empty = game.empty_coalition_value()
v_full = game.grand_coalition_value()

print(f"v(∅) = {v_empty:.4f}  (empty coalition)")
print(f"v(N) = {v_full:.4f}  (grand coalition)")
print(f"v(N) - v(∅) = {v_full - v_empty:.4f}  (expected Shapley sum)")

## Step 3: Train TN Surrogate

In [ ]:
# Create graph-aligned tensor network
tn = GraphAlignedTN(
    n_nodes=N_NODES,
    graph_structure=G,
    bond_dim=BOND_DIM,
    seed=SEED,
)
print(f"TN surrogate: {tn}")
print(f"Parameters: {tn.get_num_parameters()}")

In [ ]:
# Training configuration
train_config = TrainingConfig(
    lr=0.01,
    epochs=100,
    batch_size=32,
    n_train_samples=N_TRAIN_SAMPLES,
    n_val_samples=100,
    patience=15,
    seed=SEED,
    verbose=True,
    log_every=20,
)

print("Training TN surrogate...")
start_time = time.time()

result = train_surrogate(tn, game, config=train_config, device=device)

print(f"\nTraining completed in {time.time() - start_time:.1f}s")
print(f"Final train R²: {result.final_train_r2:.4f}")
print(f"Final val R²: {result.final_val_r2:.4f}")

## Step 4: Compute TN-SHAP-G Shapley Values

In [ ]:
print(f"Computing TN-SHAP-G Shapley values with m={M_INTERP} interpolation nodes...")
start_time = time.time()

phi_tnshap = compute_diagonal_shapley(
    tn, n_nodes=N_NODES, m=M_INTERP, device=device, verbose=True
)

print(f"\nCompleted in {time.time() - start_time:.1f}s")
print(f"Shapley sum: {np.sum(phi_tnshap):.4f}")
print(f"Expected (v(N)-v(∅)): {v_full - v_empty:.4f}")

## Step 5: Compute Exact Shapley Values (Ground Truth)

In [ ]:
def game_value_fn(S):
    """Query game value for coalition set S."""
    return game.query_coalition_set(S)

print(f"Computing exact Shapley values via enumeration (2^{N_NODES} = {2**N_NODES} coalitions)...")
start_time = time.time()

phi_exact, coalition_values = exact_shapley_fast(game_value_fn, N_NODES, verbose=True)

print(f"\nCompleted in {time.time() - start_time:.1f}s")

## Step 6: Compare TN-SHAP-G vs Exact

In [ ]:
# Compute metrics
cos_sim = cosine_similarity(phi_tnshap, phi_exact)
max_err = max_abs_error(phi_tnshap, phi_exact)
mae = np.mean(np.abs(phi_tnshap - phi_exact))

print("=" * 50)
print("O1 Shapley Value Comparison")
print("=" * 50)
print(f"Cosine similarity: {cos_sim:.6f}")
print(f"Max absolute error: {max_err:.6f}")
print(f"Mean absolute error: {mae:.6f}")
print()

# Side-by-side comparison
print(f"{'Node':>6} | {'TN-SHAP-G':>12} | {'Exact':>12} | {'Diff':>12}")
print(f"{'-'*6}-+-{'-'*12}-+-{'-'*12}-+-{'-'*12}")
for i in range(N_NODES):
    diff = phi_tnshap[i] - phi_exact[i]
    print(f"{i:>6} | {phi_tnshap[i]:>+12.6f} | {phi_exact[i]:>+12.6f} | {diff:>+12.6f}")

## Step 7: Compute O2 Interactions

In [ ]:
# Get edge pairs from graph
edge_pairs = [(min(u, v), max(u, v)) for u, v in G.edges()]
print(f"Computing O2 interactions for {len(edge_pairs)} graph edges...")

start_time = time.time()

# TN-SHAP-G O2
phi2_tnshap = compute_o2_interactions(
    tn, n_nodes=N_NODES, edge_pairs=edge_pairs, m=M_INTERP, device=device, verbose=False
)

print(f"TN-SHAP-G O2: {time.time() - start_time:.1f}s")

# Exact O2
start_time = time.time()

phi2_exact = compute_o2_exact(game_value_fn, N_NODES, edge_pairs=edge_pairs, verbose=False)

print(f"Exact O2: {time.time() - start_time:.1f}s")

In [ ]:
# Compare O2
phi2_tnshap_flat = np.array([phi2_tnshap[u, v] for u, v in edge_pairs])
phi2_exact_flat = np.array([phi2_exact[u, v] for u, v in edge_pairs])

cos_sim_o2 = cosine_similarity(phi2_tnshap_flat, phi2_exact_flat)
max_err_o2 = max_abs_error(phi2_tnshap_flat, phi2_exact_flat)

print("=" * 50)
print("O2 Shapley Interaction Comparison")
print("=" * 50)
print(f"Cosine similarity: {cos_sim_o2:.6f}")
print(f"Max absolute error: {max_err_o2:.6f}")
print()

# Show top interactions
print("Top 5 interactions (by magnitude):")
sorted_pairs = sorted(enumerate(zip(edge_pairs, phi2_tnshap_flat, phi2_exact_flat)),
                       key=lambda x: abs(x[1][1]), reverse=True)
for _, ((u, v), tn_val, exact_val) in sorted_pairs[:5]:
    print(f"  Edge ({u},{v}): TN-SHAP-G={tn_val:+.4f}, Exact={exact_val:+.4f}")

## Summary

In [ ]:
print("=" * 60)
print("TN-SHAP-G EXPERIMENT SUMMARY")
print("=" * 60)
print(f"Graph: {N_NODES} nodes, {G.number_of_edges()} edges")
print(f"TN parameters: {tn.get_num_parameters()}")
print(f"Training samples: {N_TRAIN_SAMPLES}")
print()
print(f"Surrogate validation R²: {result.final_val_r2:.4f}")
print(f"O1 Shapley cosine similarity: {cos_sim:.6f}")
print(f"O2 Interaction cosine similarity: {cos_sim_o2:.6f}")
print("=" * 60)

if cos_sim > 0.95 and cos_sim_o2 > 0.90:
    print("\n✅ SUCCESS: High-quality Shapley approximation achieved!")
else:
    print("\n⚠️ Results lower than expected. Consider increasing training samples or bond dimension.")

## Next Steps

To use TN-SHAP-G on your own data:

1. **Replace the teacher model** with your GNN or other predictor
2. **Adjust graph structure** to match your problem
3. **Tune hyperparameters**:
   - `bond_dim`: Higher for more complex interactions (4-16)
   - `n_train_samples`: More samples for larger graphs
   - `m_interpolation`: 16-32 typically sufficient

See `scripts/` for command-line tools and `README.md` for more details.